# USD/JPY 择时损益时间序列

**创建时间**: 2026-01-11

**描述**: 画三条线 - USD损益、JPY损益、两者合计

**数据来源**:
- 周五预锁价交易明细_新.xlsx: 2025年1-10月明细数据
- 周五预锁价_11月-12月.xlsx: 2025年11-12月汇总数据

## 1. 环境设置

In [ ]:
import sys
from pathlib import Path

# 路径设置
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(0, str(ROOT))

# 数据目录
SHARED_DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"数据目录: {SHARED_DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

## 2. 数据加载

In [ ]:
# 查看数据目录
list(SHARED_DATA_DIR.glob("*"))

In [ ]:
# 加载1-10月明细数据
df_detail = pd.read_excel(SHARED_DATA_DIR / "周五预锁价交易明细_新.xlsx")
print(f"1-10月明细: {df_detail.shape}")
print(f"日期范围: {df_detail['交易日期'].min()} ~ {df_detail['交易日期'].max()}")
df_detail.head()

In [ ]:
# 加载11-12月汇总数据
df_summary = pd.read_excel(SHARED_DATA_DIR / "周五预锁价_11月-12月.xlsx")
print(f"11-12月汇总: {df_summary.shape}")
df_summary.head()

## 3. 数据处理

In [ ]:
# ========== 处理1-10月明细数据 ==========
# 用日期-1天得到周五日期（所有交易归到周五）
df1 = df_detail[['周六日期', '币种对', '执行损益_CNY', '平仓损益']].copy()
df1.columns = ['周六日期', '币种对', '执行损益', '平仓损益']
df1['周六日期'] = pd.to_datetime(df1['周六日期'])
df1['周五日期'] = df1['周六日期'] - pd.Timedelta(days=1)
df1['总损益'] = df1['执行损益'] + df1['平仓损益']

# 按周五日期和币种汇总
df1_agg = df1.groupby(['周五日期', '币种对'])['总损益'].sum().reset_index()
df1_agg = df1_agg.rename(columns={'周五日期': '日期'})

print(f"1-10月处理后: {df1_agg.shape}")
print(f"日期范围: {df1_agg['日期'].min()} ~ {df1_agg['日期'].max()}")
print(f"周五数量: {df1_agg['日期'].nunique()}")
df1_agg.head(10)

In [ ]:
# ========== 处理11-12月汇总数据==========
# 只保留五(4)和六(5)的交易，周六归到周五
df2 = df_summary[df_summary['交易日期'] != '-'].copy()
df2['交易日期'] = pd.to_datetime(df2['交易日期'], format='%Y%m%d')
df2['weekday'] = df2['交易日期'].dt.dayofweek  # 0=Mon, 4=Fri, 5=Sat

# 只保留周五和周六
df2 = df2[df2['weekday'].isin([4, 5])].copy()

# 周六归到周五（日期-1天）
df2['周五日期'] = df2.apply(lambda r: r['交易日期'] - pd.Timedelta(days=1) if r['weekday'] == 5 else r['交易日期'], axis=1)

df2 = df2[['周五日期', '币种对', '择时执行损益', '择时平仓损益']].copy()
df2.columns = ['日期', '币种对', '执行损益', '平仓损益']
df2['总损益'] = df2['执行损益'] + df2['平仓损益']

# 按周五日期和币种汇总
df2_agg = df2.groupby(['日期', '币种对'])['总损益'].sum().reset_index()

print(f"11-12月处理后: {df2_agg.shape}")
print(f"日期范围: {df2_agg['日期'].min()} ~ {df2_agg['日期'].max()}")
print(f"周五数量: {df2_agg['日期'].nunique()}")
df2_agg

In [ ]:
# ========== 合并数据 ==========
df_all = pd.concat([df1_agg, df2_agg], ignore_index=True)

# 按日期和币种再次汇总（处理可能的重叠）
df_all = df_all.groupby(['日期', '币种对'])['总损益'].sum().reset_index()
df_all = df_all.sort_values('日期').reset_index(drop=True)

print(f"合并后: {df_all.shape}")
print(f"日期范围: {df_all['日期'].min()} ~ {df_all['日期'].max()}")
print(f"周五数量: {df_all['日期'].nunique()}")
df_all

In [ ]:
# ========== 分离USD和JPY，按日期透视 ==========
# 透视表：日期为行，币种为列
pivot = df_all.pivot_table(index='日期', columns='币种对', values='总损益', aggfunc='sum').fillna(0)
pivot = pivot.reset_index()

# 确保列名正确
if 'USDCNH' not in pivot.columns:
    pivot['USDCNH'] = 0
if 'JPYCNH' not in pivot.columns:
    pivot['JPYCNH'] = 0

# 重命名
data = pivot.rename(columns={'USDCNH': 'USD损益', 'JPYCNH': 'JPY损益'})

# 按日期排序
data = data.sort_values('日期').reset_index(drop=True)

# 计算累计损益
data['USD累计'] = data['USD损益'].cumsum()
data['JPY累计'] = data['JPY损益'].cumsum()
data['总计累计'] = data['USD累计'] + data['JPY累计']

print(f"最终数据 {data.shape}")
print(f"周五数量: {len(data)}")
data

In [ ]:
# ========== 汇总统计 ==========
print("="*60)
print("汇总统计")
print("="*60)
print(f"数据范围: {data['日期'].min().strftime('%Y-%m-%d')} ~ {data['日期'].max().strftime('%Y-%m-%d')}")
print(f"周五数量: {len(data)}")
print("-"*60)
print(f"USD总损益' {data['USD损益'].sum():>15,.2f}")
print(f"JPY总损益' {data['JPY损益'].sum():>15,.2f}")
print(f"合计总损益' {data['USD损益'].sum() + data['JPY损益'].sum():>14,.2f}")
print("="*60)

## 4. 可视化

In [ ]:
# 创建累计损益时间序列图（Excel风格）
fig = go.Figure()

# 获取最后一个数据点的值（用于标注）
last_idx = len(data) - 1
last_date = data['日期'].iloc[last_idx]
last_total = data['总计累计'].iloc[last_idx]
last_usd = data['USD累计'].iloc[last_idx]
last_jpy = data['JPY累计'].iloc[last_idx]

# 美元 - 蓝色
fig.add_trace(go.Scatter(
    x=data['日期'],
    y=data['USD累计'],
    mode='lines+markers',
    name='美元',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=6, symbol='diamond')
))

# 日元 - 红棕色
fig.add_trace(go.Scatter(
    x=data['日期'],
    y=data['JPY累计'],
    mode='lines+markers',
    name='日元',
    line=dict(color='#C55A11', width=2),
    marker=dict(size=6, symbol='diamond')
))

# 合计 - 橙色
fig.add_trace(go.Scatter(
    x=data['日期'],
    y=data['总计累计'],
    mode='lines+markers',
    name='合计',
    line=dict(color='#ED7D31', width=2),
    marker=dict(size=6, symbol='diamond')
))

# 在最后一个点添加数值标注
fig.add_annotation(
    x=last_date, y=last_total,
    text=f'¥{last_total:,.0f}',
    showarrow=False,
    font=dict(color='#ED7D31', size=11, family='Arial'),
    xanchor='left', xshift=8
)

# 零线
fig.add_hline(y=0, line_width=1, line_color='#808080')

fig.update_layout(
    title=dict(
        text='周五预锁价累计损益',
        font=dict(size=18, color='#333333', family='Microsoft YaHei'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title=dict(text='日期', font=dict(size=12)),
        tickformat='%Y-%m-%d',
        tickangle=-45,
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True,
        linecolor='#CCCCCC',
        linewidth=1,
        mirror=True
    ),
    yaxis=dict(
        title=dict(text='累计损益（人民币元）', font=dict(size=12)),
        tickprefix='¥',
        tickformat=',.0f',
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True,
        zeroline=False,
        linecolor='#CCCCCC',
        linewidth=1,
        mirror=True
    ),
    height=500,
    width=1000,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.98,
        xanchor='left',
        x=0.02,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC',
        borderwidth=1,
        font=dict(size=11)
    ),
    hovermode='x unified',
    hoverlabel=dict(
        bgcolor='white',
        font_size=11,
        font_family='Microsoft YaHei'
    ),
    margin=dict(l=80, r=100, t=60, b=100)
)

fig.show()

In [ ]:
# 保存图表
fig.write_html(str(OUTPUT_DIR / "pnl_timeseries.html"))
print(f"已保存: {OUTPUT_DIR / 'pnl_timeseries.html'}")

## 5. 每周损益图表

In [ ]:
# 每周损益柱状图
fig2 = make_subplots(rows=2, cols=1, 
                     subplot_titles=('每周损益', '累计损益'),
                     vertical_spacing=0.12,
                     row_heights=[0.4, 0.6])

# 上图：每周损益柱状图
fig2.add_trace(go.Bar(x=data['日期'], y=data['USD损益'], name='USD每周', marker_color='#4472C4', opacity=0.8), row=1, col=1)
fig2.add_trace(go.Bar(x=data['日期'], y=data['JPY损益'], name='JPY每周', marker_color='#C55A11', opacity=0.8), row=1, col=1)

# 下图：累计损益
fig2.add_trace(go.Scatter(x=data['日期'], y=data['USD累计'], mode='lines+markers', name='USD累计', line=dict(color='#4472C4', width=2), marker=dict(size=5)), row=2, col=1)
fig2.add_trace(go.Scatter(x=data['日期'], y=data['JPY累计'], mode='lines+markers', name='JPY累计', line=dict(color='#C55A11', width=2), marker=dict(size=5)), row=2, col=1)
fig2.add_trace(go.Scatter(x=data['日期'], y=data['总计累计'], mode='lines+markers', name='合计', line=dict(color='#ED7D31', width=2), marker=dict(size=5)), row=2, col=1)

fig2.add_hline(y=0, line_dash='solid', line_color='#808080', line_width=1, row=1, col=1)
fig2.add_hline(y=0, line_dash='solid', line_color='#808080', line_width=1, row=2, col=1)

fig2.update_layout(
    title=dict(
        text='周五预锁价损益分析(2025全年)',
        font=dict(size=18, color='#333333'),
        x=0.5, xanchor='center'
    ),
    height=700,
    width=1000,
    barmode='group',
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig2.update_xaxes(tickformat='%Y-%m-%d', tickangle=-45, gridcolor='#E8E8E8')
fig2.update_yaxes(tickprefix='¥', tickformat=',.0f', gridcolor='#E8E8E8')

fig2.show()

## 6. 策略价值对比分析

对比择时策略 vs 固定时间点买入策略的累计收益差异

In [ ]:
# 加载15分钟K线数据和2点参考价格
df_15min = pd.read_excel(SHARED_DATA_DIR / "USDCNH_15min.xlsx")
df_ref = pd.read_excel(SHARED_DATA_DIR / "周六2点参考价格表 (1).xlsx", skiprows=1)
df_ref.columns = ['周六日期', '币种对', '全民_2点汇率', '李刚_2点汇率', 'BBG_基准价', 
                  '全民vs BBG(bps)', '李刚vs BBG(bps)', '最终参考价', '选择逻辑']

print(f"15分钟数据: {df_15min.shape}")
print(f"2点参考价格: {df_ref.shape}")
print(f"\n15分钟数据时间范围: {df_15min['Date'].min()} ~ {df_15min['Date'].max()}")

In [ ]:
# 处理15分钟数据
df_15min['Date'] = pd.to_datetime(df_15min['Date'])
df_15min['weekday'] = df_15min['Date'].dt.dayofweek
df_15min['hour'] = df_15min['Date'].dt.hour
df_15min['minute'] = df_15min['Date'].dt.minute

# 周五晚上21:00-23:59的数据
fri_night = df_15min[(df_15min['weekday'] == 4) & (df_15min['hour'] >= 21)].copy()
fri_night['fri_date'] = fri_night['Date'].dt.date

# 周六凌晨00:00-02:00的数据
sat_morning = df_15min[(df_15min['weekday'] == 5) & (df_15min['hour'] <= 2)].copy()
sat_morning['fri_date'] = (sat_morning['Date'] - pd.Timedelta(days=1)).dt.date

# 合并
price_data = pd.concat([fri_night, sat_morning]).sort_values('Date')
print(f"周五晚+周六凌晨数据: {len(price_data)}")
print(f"覆盖周五数量: {price_data['fri_date'].nunique()}")

In [ ]:
# 定义固定买入时间点(北京时间)
# 21:30, 22:00, 22:30, 23:00, 23:30, 00:00, 00:30
buy_times = [
    (21, 30, '21:30'),
    (22, 0, '22:00'),
    (22, 30, '22:30'),
    (23, 0, '23:00'),
    (23, 30, '23:30'),
    (0, 0, '00:00'),
    (0, 30, '00:30'),
]

# 卖出时间: 02:00
sell_hour, sell_minute = 2, 0

# 提取每周各时间点的价格
results = []

for fri_date in sorted(price_data['fri_date'].unique()):
    week_data = price_data[price_data['fri_date'] == fri_date]
    
    # 获取02:00卖出价格
    sell_data = week_data[(week_data['hour'] == sell_hour) & (week_data['minute'] == sell_minute)]
    if len(sell_data) == 0:
        continue
    sell_price = sell_data.iloc[0]['Open']
    
    row = {'fri_date': fri_date, 'sell_price_02:00': sell_price}
    
    # 获取各买入时间点价格
    for h, m, label in buy_times:
        buy_data = week_data[(week_data['hour'] == h) & (week_data['minute'] == m)]
        if len(buy_data) > 0:
            row[f'buy_{label}'] = buy_data.iloc[0]['Open']
        else:
            row[f'buy_{label}'] = np.nan
    
    results.append(row)

df_prices = pd.DataFrame(results)
print(f"提取到{len(df_prices)} 周的价格数据")
df_prices.head(10)

In [ ]:
# 计算各固定策略的损益 (bps)
# 损益 = (卖出价 - 买入价) / 买入价 * 10000 bps
# 买入USD（卖CNH），汇率上涨赚钱

for h, m, label in buy_times:
    col_buy = f'buy_{label}'
    col_pnl = f'pnl_{label}'
    # 汇率上涨赚钱: (卖出价 - 买入价) / 买入价 * 10000
    df_prices[col_pnl] = (df_prices['sell_price_02:00'] - df_prices[col_buy]) / df_prices[col_buy] * 10000

# 计算固定策略平均
pnl_cols = [f'pnl_{label}' for _, _, label in buy_times]
df_prices['pnl_avg'] = df_prices[pnl_cols].mean(axis=1)

print("各策略损益统计(bps):")
for col in pnl_cols + ['pnl_avg']:
    print(f"  {col}: 均值={df_prices[col].mean():.2f}, 累计={df_prices[col].sum():.2f}")

In [ ]:
# 获取每周真实USDCNH交易量
# 1-10月明细数据提取
df_vol_1 = df_detail[df_detail['币种对'] == 'USDCNH'][['周六日期', 'Volume 成交量']].copy()
df_vol_1['周六日期'] = pd.to_datetime(df_vol_1['周六日期'])
df_vol_1['fri_date'] = (df_vol_1['周六日期'] - pd.Timedelta(days=1)).dt.date
df_vol_1 = df_vol_1.groupby('fri_date')['Volume 成交量'].sum().reset_index()
df_vol_1.columns = ['fri_date', 'volume']

# 11-12月汇总数据提取交易量（列名是择时执行交易量'择时平仓交易量）
df_vol_2 = df_summary[df_summary['币种对'] == 'USDCNH'][['交易日期', '择时执行交易量', '择时平仓交易量']].copy()
df_vol_2 = df_vol_2[df_vol_2['交易日期'] != '-']
df_vol_2['交易量'] = df_vol_2['择时执行交易量'].fillna(0) + df_vol_2['择时平仓交易量'].fillna(0)
df_vol_2['交易日期'] = pd.to_datetime(df_vol_2['交易日期'], format='%Y%m%d')
df_vol_2['weekday'] = df_vol_2['交易日期'].dt.dayofweek
df_vol_2 = df_vol_2[df_vol_2['weekday'].isin([4, 5])].copy()
df_vol_2['fri_date'] = df_vol_2.apply(lambda r: (r['交易日期'] - pd.Timedelta(days=1)).date() if r['weekday'] == 5 else r['交易日期'].date(), axis=1)
df_vol_2 = df_vol_2.groupby('fri_date')['交易量'].sum().reset_index()
df_vol_2.columns = ['fri_date', 'volume']

# 合并交易量
df_volume = pd.concat([df_vol_1, df_vol_2]).groupby('fri_date')['volume'].sum().reset_index()
print(f"交易量数据 {len(df_volume)} 周")
print(f"交易量范围: {df_volume['volume'].min():,.0f} ~ {df_volume['volume'].max():,.0f}")
df_volume.head()


In [ ]:
# 合并价格数据和交易量
df_prices = pd.merge(df_prices, df_volume, on='fri_date', how='left')

# 用真实交易量计算各固定策略的CNY损益
# 损益(CNY) = 交易量(USD) * (卖出价 - 买入价)
for h, m, label in buy_times:
    col_buy = f'buy_{label}'
    col_pnl_cny = f'pnl_cny_{label}'
    df_prices[col_pnl_cny] = df_prices['volume'] * (df_prices['sell_price_02:00'] - df_prices[col_buy])

# 计算固定策略平均CNY损益
pnl_cny_cols = [f'pnl_cny_{label}' for _, _, label in buy_times]
df_prices['pnl_cny_avg'] = df_prices[pnl_cny_cols].mean(axis=1)

print("各固定策略CNY损益统计:")
for col in pnl_cny_cols + ['pnl_cny_avg']:
    print(f"  {col}: 累计={df_prices[col].sum():,.0f} CNY")

In [ ]:
# 获取择时策略的USDCNH损益数据（已是CNY）
df_strategy = data[['日期', 'USD损益']].copy()
df_strategy['fri_date'] = df_strategy['日期'].dt.date
df_strategy['strategy_pnl_cny'] = df_strategy['USD损益']

print(f"择时策略数据: {len(df_strategy)} 周")
print(f"择时策略累计损益: {df_strategy['strategy_pnl_cny'].sum():,.0f} CNY")
df_strategy.head()

In [ ]:
# 合并两个数据集
df_compare = pd.merge(
    df_prices,
    df_strategy[['fri_date', 'strategy_pnl_cny']],
    on='fri_date',
    how='inner'
)

print(f"匹配到{len(df_compare)} 周的对比数据")
print(f"日期范围: {df_compare['fri_date'].min()} ~ {df_compare['fri_date'].max()}")

# 计算累计损益 (万元)
df_compare = df_compare.sort_values('fri_date').reset_index(drop=True)
df_compare['week_num'] = range(1, len(df_compare) + 1)

# 累计各策略损益（转换为万元）
for col in pnl_cny_cols + ['pnl_cny_avg', 'strategy_pnl_cny']:
    df_compare[f'{col}_cum'] = df_compare[col].cumsum() / 10000  # 转换为万元

df_compare.head()

In [ ]:
# 绘制策略对比图（CNY万元）
fig3 = go.Figure()

# 固定策略 - 灰色虚线
for h, m, label in buy_times:
    col = f'pnl_cny_{label}_cum'
    fig3.add_trace(go.Scatter(
        x=df_compare['week_num'],
        y=df_compare[col],
        mode='lines',
        name=f'固定策略-{label}',
        line=dict(color='#999999', width=1, dash='dash'),
        opacity=0.7
    ))

# 固定策略平均 - 橙色实线
fig3.add_trace(go.Scatter(
    x=df_compare['week_num'],
    y=df_compare['pnl_cny_avg_cum'],
    mode='lines+markers',
    name='固定策略平均',
    line=dict(color='#FFA500', width=2.5),
    marker=dict(size=4)
))

# 择时策略 - 绿色实线
fig3.add_trace(go.Scatter(
    x=df_compare['week_num'],
    y=df_compare['strategy_pnl_cny_cum'],
    mode='lines+markers',
    name='择时策略',
    line=dict(color='#2CA02C', width=2.5),
    marker=dict(size=4)
))

# 零线
fig3.add_hline(y=0, line_width=1, line_color='#808080')

fig3.update_layout(
    title=dict(
        text='USDCNH 择时策略 vs 固定时间点策略累计收益对比',
        font=dict(size=16, color='#333333'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='周数',
        tickfont=dict(size=10),
        gridcolor='#E8E8E8'
    ),
    yaxis=dict(
        title='累计收益（万元）',
        tickfont=dict(size=10),
        gridcolor='#E8E8E8'
    ),
    height=500,
    width=1100,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='v',
        yanchor='middle',
        y=0.5,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC',
        borderwidth=1,
        font=dict(size=9)
    ),
    hovermode='x unified',
    margin=dict(l=60, r=150, t=60, b=60)
)

fig3.show()


In [ ]:
# 汇总统计
print("="*70)
print("策略对比汇总(USDCNH) - 使用真实交易量")
print("="*70)
print(f"数据周数: {len(df_compare)}")
print(f"日期范围: {df_compare['fri_date'].min()} ~ {df_compare['fri_date'].max()}")
print("-"*70)
print(f"{'策略':<20} {'累计收益(万元)':<18} {'周均收益(元)':<15}")
print("-"*70)

for h, m, label in buy_times:
    col = f'pnl_cny_{label}'
    total = df_compare[col].sum() / 10000
    avg = df_compare[col].mean()
    print(f"固定略-{label:<10} {total:>14,.2f} {avg:>14,.0f}")

print("-"*70)
print(f"{'固定策略平均':<18} {df_compare['pnl_cny_avg'].sum()/10000:>14,.2f} {df_compare['pnl_cny_avg'].mean():>14,.0f}")
print(f"{'择时策略':<18} {df_compare['strategy_pnl_cny'].sum()/10000:>14,.2f} {df_compare['strategy_pnl_cny'].mean():>14,.0f}")
print("="*70)

# 策略增量
strategy_total = df_compare['strategy_pnl_cny'].sum()
avg_total = df_compare['pnl_cny_avg'].sum()
print(f"\n择时策略相对固定策略平均的增量: {(strategy_total - avg_total)/10000:,.2f} 万元")

## 7. 周一反平策略损益分析

周一反平策略的累计损益表现

In [ ]:
# 加载周一反平数据 (1-10元 =
df_monday = pd.read_excel(SHARED_DATA_DIR / "周一反平.xlsx")
print(f"周一反平数据: {df_monday.shape}")
print(df_monday.columns.tolist())
df_monday.head()

In [ ]:
# 处理周一反平数据
# 数据日期格式: 2025-10-27~2025-11-02，取周一日期（范围的第一天）
df_mon1 = df_monday.copy()
df_mon1['周一日期'] = df_mon1['数据日期'].str.split('~').str[0]
df_mon1['周一日期'] = pd.to_datetime(df_mon1['周一日期'])
df_mon1 = df_mon1[['周一日期', '交易收益']].copy()
df_mon1.columns = ['日期', '损益']

# 过滤掉11月及以后的数据（避免11-12月数据重复）
df_mon1 = df_mon1[df_mon1['日期'] < '2025-11-01']

print(f"1-10月周一反平: {len(df_mon1)} 条")
print(f"日期范围: {df_mon1['日期'].min()} ~ {df_mon1['日期'].max()}")
df_mon1.head()

In [ ]:
# 11-12月数据提取周一的损益
df_mon2 = df_summary[df_summary['币种对'] == 'USDCNH'].copy()
df_mon2 = df_mon2[df_mon2['交易日期'] != '-']
df_mon2['交易日期'] = pd.to_datetime(df_mon2['交易日期'], format='%Y%m%d')
df_mon2['weekday'] = df_mon2['交易日期'].dt.dayofweek

# 只保留周一 (weekday=0)
df_mon2 = df_mon2[df_mon2['weekday'] == 0].copy()
df_mon2['损益'] = df_mon2['择时执行损益'].fillna(0) + df_mon2['择时平仓损益'].fillna(0)
df_mon2 = df_mon2[['交易日期', '损益']].copy()
df_mon2.columns = ['日期', '损益']

print(f"11-12月周一反平: {len(df_mon2)} 条")
print(f"日期范围: {df_mon2['日期'].min()} ~ {df_mon2['日期'].max()}")
df_mon2

In [ ]:
# 合并周一反平数据
df_monday_all = pd.concat([df_mon1, df_mon2], ignore_index=True)
df_monday_all = df_monday_all.sort_values('日期').reset_index(drop=True)

# 计算累计损益（万元）
df_monday_all['累计损益'] = df_monday_all['损益'].cumsum()
df_monday_all['累计损益_万元'] = df_monday_all['累计损益'] / 10000

print(f"周一反平合计: {len(df_monday_all)} 周")
print(f"日期范围: {df_monday_all['日期'].min()} ~ {df_monday_all['日期'].max()}")
print(f"累计损益: {df_monday_all['累计损益'].iloc[-1]:,.0f} 元 = {df_monday_all['累计损益_万元'].iloc[-1]:.1f}万元")
df_monday_all

In [ ]:
# 绘制周一反平策略累计损益图
fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x=df_monday_all['日期'],
    y=df_monday_all['累计损益_万元'],
    mode='lines+markers',
    name='周一反平策略',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=4)
))

# 零线
fig4.add_hline(y=0, line_width=1, line_color='#808080')

# 最终值标注
last_date = df_monday_all['日期'].iloc[-1]
last_value = df_monday_all['累计损益_万元'].iloc[-1]
fig4.add_annotation(
    x=last_date, y=last_value,
    text=f'{last_value:.1f}万元',
    showarrow=False,
    font=dict(color='#4472C4', size=12),
    xanchor='left', xshift=10
)

fig4.update_layout(
    title=dict(
        text='USDCNH: 周一反平策略<br><sup>累计损益表现</sup>',
        font=dict(size=16, color='#4472C4'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='交易月份',
        tickformat='%Y-%m',
        tickangle=-30,
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True,
        linecolor='#CCCCCC',
        linewidth=1,
        mirror=True
    ),
    yaxis=dict(
        title='累计损益（万元）',
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True,
        linecolor='#CCCCCC',
        linewidth=1,
        mirror=True
    ),
    height=450,
    width=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(l=60, r=80, t=80, b=60)
)

fig4.show()

In [ ]:
# 汇总统计
print("="*50)
print("周一反平策略汇总")
print("="*50)
print(f"数据周数: {len(df_monday_all)}")
print(f"日期范围: {df_monday_all['日期'].min().strftime('%Y-%m-%d')} ~ {df_monday_all['日期'].max().strftime('%Y-%m-%d')}")
print(f"累计损益: {df_monday_all['累计损益'].iloc[-1]:,.0f} 元")
print(f"累计损益: {df_monday_all['累计损益_万元'].iloc[-1]:.1f} 万元")
print(f"周均损益: {df_monday_all['损益'].mean():,.0f} 元")
print(f"胜率: {(df_monday_all['损益'] > 0).sum() / len(df_monday_all) * 100:.1f}%")
print("="*50)

## 8. 2025年择时策略综合表现

周五预锁价 + 周一反平 两大策略的累计损益

In [ ]:
# 准备周五预锁价数据（USD+JPY合计）
df_friday = data[['日期', 'USD损益', 'JPY损益']].copy()
df_friday['损益'] = df_friday['USD损益'] + df_friday['JPY损益']
df_friday['累计损益_万元'] = df_friday['损益'].cumsum() / 10000
df_friday = df_friday.rename(columns={'日期': '日期'})

print(f"周五预锁价: {len(df_friday)} 周, 累计 {df_friday['累计损益_万元'].iloc[-1]:.1f}万元")
print(f"周一反平: {len(df_monday_all)} 周, 累计 {df_monday_all['累计损益_万元'].iloc[-1]:.1f}万元")

In [ ]:
# 绘制组合元 =- 上下布局，共享X轴
from plotly.subplots import make_subplots

# 准备周五预锁价数据
df_friday = data[['日期', 'USD损益', 'JPY损益']].copy()
df_friday['合计损益'] = df_friday['USD损益'] + df_friday['JPY损益']
df_friday['USD累计_万元'] = df_friday['USD损益'].cumsum() / 10000
df_friday['JPY累计_万元'] = df_friday['JPY损益'].cumsum() / 10000
df_friday['合计累计_万元'] = df_friday['合计损益'].cumsum() / 10000

fig5 = make_subplots(
    rows=2, cols=1,
    subplot_titles=('<b>周五预锁价策略</b>', '<b>周一反平策略</b>'),
    vertical_spacing=0.10,
    shared_xaxes=True
)

# 上图：周五预锁价 - USD
fig5.add_trace(go.Scatter(
    x=df_friday['日期'],
    y=df_friday['USD累计_万元'],
    mode='lines+markers',
    name='美元',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=4),
    legend='legend1'
), row=1, col=1)

# 上图：周五预锁价 - JPY
fig5.add_trace(go.Scatter(
    x=df_friday['日期'],
    y=df_friday['JPY累计_万元'],
    mode='lines+markers',
    name='日元',
    line=dict(color='#C55A11', width=2),
    marker=dict(size=4),
    legend='legend1'
), row=1, col=1)

# 上图：周五预锁价 - 合计
fig5.add_trace(go.Scatter(
    x=df_friday['日期'],
    y=df_friday['合计累计_万元'],
    mode='lines+markers',
    name='合计',
    line=dict(color='#ED7D31', width=2.5),
    marker=dict(size=5),
    legend='legend1'
), row=1, col=1)

# 下图：周一反平 - USD
fig5.add_trace(go.Scatter(
    x=df_monday_all['日期'],
    y=df_monday_all['累计损益_万元'],
    mode='lines+markers',
    name='美元',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=4),
    legend='legend2'
), row=2, col=1)

# 零线
fig5.add_hline(y=0, line_width=1, line_color='#808080', row=1, col=1)
fig5.add_hline(y=0, line_width=1, line_color='#808080', row=2, col=1)

# 最终值
fri_total_final = df_friday['合计累计_万元'].iloc[-1]
mon_final = df_monday_all['累计损益_万元'].iloc[-1]

# 上图标注
fig5.add_annotation(
    x=df_friday['日期'].iloc[-1], y=fri_total_final,
    text=f'{fri_total_final:.1f}万元',
    showarrow=False, font=dict(color='#ED7D31', size=11),
    xanchor='left', xshift=8,
    row=1, col=1
)

# 下图标注
fig5.add_annotation(
    x=df_monday_all['日期'].iloc[-1], y=mon_final,
    text=f'{mon_final:.1f}万元',
    showarrow=False, font=dict(color='#4472C4', size=11),
    xanchor='left', xshift=8,
    row=2, col=1
)

# 周一反平三月上线标注
fig5.add_annotation(
    x=pd.Timestamp('2025-03-01'), y=0,
    text='3月上线',
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=1.5,
    arrowcolor='#666666',
    ax=0, ay=-35,
    font=dict(color='#666666', size=10),
    xanchor='center',
    yanchor='top',
    row=2, col=1
)

# X轴设置（只在底部显示）
fig5.update_xaxes(tickformat='%Y-%m', tickangle=-30, tickfont=dict(size=10), gridcolor='#E8E8E8', row=2, col=1)
fig5.update_xaxes(showticklabels=False, gridcolor='#E8E8E8', row=1, col=1)

# Y轴设元 =- 加粗放大
fig5.update_yaxes(
    title_text='<b>累计损益（万元）</b>',
    title_font=dict(size=13),
    tickfont=dict(size=10),
    gridcolor='#E8E8E8',
    row=1, col=1
)
fig5.update_yaxes(
    title_text='<b>累计损益（万元）</b>',
    title_font=dict(size=13),
    tickfont=dict(size=10),
    gridcolor='#E8E8E8',
    row=2, col=1
)

fig5.update_layout(
    height=650,
    width=950,
    plot_bgcolor='white',
    paper_bgcolor='white',
    # 第一个图例（上图）
    legend1=dict(
        orientation='v',
        yanchor='top',
        y=0.98,
        xanchor='left',
        x=1.02,
        font=dict(size=10),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC',
        borderwidth=1
    ),
    # 第二个图例（下图）
    legend2=dict(
        orientation='v',
        yanchor='top',
        y=0.45,
        xanchor='left',
        x=1.02,
        font=dict(size=10),
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC',
        borderwidth=1
    ),
    margin=dict(l=70, r=100, t=50, b=60)
)

# 子图标题样式 - 放大加粗
for ann in fig5.layout.annotations:
    if '周五预锁价策略' in str(ann.text) or '周一反平' in str(ann.text):
        ann.font = dict(size=15, color='#333333')

fig5.show()

## 9. 周五预锁价策略阶段分析

探索期 → 调整期 → 扩张期

In [ ]:
# 周五预锁价阶段分析图
fig6 = go.Figure()

# 准备数据
df_phase = data[['日期', 'USD损益', 'JPY损益']].copy()
df_phase['合计损益'] = df_phase['USD损益'] + df_phase['JPY损益']
df_phase['USD累计'] = df_phase['USD损益'].cumsum()
df_phase['JPY累计'] = df_phase['JPY损益'].cumsum()
df_phase['合计累计'] = df_phase['合计损益'].cumsum()

# 阶段分界点
phase1_end = pd.Timestamp('2025-04-01')  # 探索期结束
phase2_end = pd.Timestamp('2025-06-01')  # 调整期结束

# 获取数据范围
x_min = df_phase['日期'].min()
x_max = df_phase['日期'].max()
y_min = min(df_phase['USD累计'].min(), df_phase['JPY累计'].min(), df_phase['合计累计'].min())
y_max = max(df_phase['USD累计'].max(), df_phase['JPY累计'].max(), df_phase['合计累计'].max())
y_padding = (y_max - y_min) * 0.1

# 添加阶段背景色
# 探索期 - 浅黄色
fig6.add_vrect(
    x0=x_min, x1=phase1_end,
    fillcolor='rgba(255, 230, 180, 0.4)',
    layer='below', line_width=0
)
# 调整期 - 浅灰色
fig6.add_vrect(
    x0=phase1_end, x1=phase2_end,
    fillcolor='rgba(220, 220, 220, 0.4)',
    layer='below', line_width=0
)
# 扩张期 - 浅绿色
fig6.add_vrect(
    x0=phase2_end, x1=x_max + pd.Timedelta(days=10),
    fillcolor='rgba(200, 240, 200, 0.4)',
    layer='below', line_width=0
)

# 添加分界线
fig6.add_vline(x=phase1_end, line_width=1.5, line_dash='dash', line_color='#888888')
fig6.add_vline(x=phase2_end, line_width=1.5, line_dash='dash', line_color='#888888')

# 美元 - 蓝色
fig6.add_trace(go.Scatter(
    x=df_phase['日期'],
    y=df_phase['USD累计'],
    mode='lines+markers',
    name='美元',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=4)
))

# 日元 - 红棕色
fig6.add_trace(go.Scatter(
    x=df_phase['日期'],
    y=df_phase['JPY累计'],
    mode='lines+markers',
    name='日元',
    line=dict(color='#C55A11', width=2),
    marker=dict(size=4)
))

# 合计 - 橙色
fig6.add_trace(go.Scatter(
    x=df_phase['日期'],
    y=df_phase['合计累计'],
    mode='lines+markers',
    name='合计',
    line=dict(color='#ED7D31', width=2.5),
    marker=dict(size=5)
))

# 零线
fig6.add_hline(y=0, line_width=1, line_color='#808080')

# 阶段标签
label_y = y_max + y_padding * 0.3
fig6.add_annotation(
    x=pd.Timestamp('2025-02-15'), y=label_y,
    text='<b>探索期</b>',
    showarrow=False,
    font=dict(size=13, color='#B8860B'),
    bgcolor='rgba(255,230,180,0.8)',
    borderpad=4
)
fig6.add_annotation(
    x=pd.Timestamp('2025-05-01'), y=label_y,
    text='<b>调整期</b>',
    showarrow=False,
    font=dict(size=13, color='#666666'),
    bgcolor='rgba(220,220,220,0.8)',
    borderpad=4
)
fig6.add_annotation(
    x=pd.Timestamp('2025-09-01'), y=label_y,
    text='<b>扩张期</b>',
    showarrow=False,
    font=dict(size=13, color='#228B22'),
    bgcolor='rgba(200,240,200,0.8)',
    borderpad=4
)

# 最终值标注
final_value = df_phase['合计累计'].iloc[-1]
fig6.add_annotation(
    x=df_phase['日期'].iloc[-1], y=final_value,
    text=f'<b>¥{final_value:,.0f}</b>',
    showarrow=False,
    font=dict(size=12, color='#ED7D31'),
    xanchor='left', xshift=8,
    bgcolor='rgba(255,255,255,0.8)'
)

fig6.update_layout(
    title=dict(
        text='<b>周五预锁价累计损益</b>',
        font=dict(size=16, color='#333333'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='<b>日期</b>',
        title_font=dict(size=12),
        tickformat='%Y-%m-%d',
        tickangle=-30,
        tickfont=dict(size=9),
        gridcolor='#E8E8E8',
        showgrid=True
    ),
    yaxis=dict(
        title='<b>累计损益（人民币元）</b>',
        title_font=dict(size=12),
        tickprefix='¥',
        tickformat=',.0f',
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True
    ),
    height=500,
    width=1050,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='v',
        yanchor='middle',
        y=0.5,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#CCCCCC',
        borderwidth=1,
        font=dict(size=10)
    ),
    margin=dict(l=80, r=100, t=60, b=80)
)

fig6.show()

## 10. 交易量稳步增长

周五预锁价策略交易量按月汇总（美元+日元，人民币计价）

In [ ]:
# 计算每月交易量（人民币计价）
# 1-10月从明细数据
df_vol_detail = df_detail[['周六日期', '币种对', 'Volume 成交量', 'Price 成交价格']].copy()
df_vol_detail['周六日期'] = pd.to_datetime(df_vol_detail['周六日期'])
df_vol_detail['月份'] = df_vol_detail['周六日期'].dt.to_period('M')

# 计算人民币交易量：USD直接乘汇率，JPY需要除100再乘汇率
df_vol_detail['交易量_CNY'] = df_vol_detail.apply(
    lambda r: r['Volume 成交量'] * r['Price 成交价格'] if r['币种对'] == 'USDCNH' 
              else r['Volume 成交量'] * r['Price 成交价格'] / 100,  # JPYCNH报价元 =00日元
    axis=1
)

# 按月汇总
vol_monthly_1 = df_vol_detail.groupby('月份')['交易量_CNY'].sum().reset_index()
vol_monthly_1['月份'] = vol_monthly_1['月份'].dt.to_timestamp()

print(f"1-10月交易量数据: {len(vol_monthly_1)} 个月")
vol_monthly_1

In [ ]:
# 11-12月从汇总数据（只有周五周六的）
df_vol_sum = df_summary[df_summary['交易日期'] != '-'].copy()
df_vol_sum['交易日期'] = pd.to_datetime(df_vol_sum['交易日期'], format='%Y%m%d')
df_vol_sum['weekday'] = df_vol_sum['交易日期'].dt.dayofweek

# 只保留周五和周六（排除周一反平）
df_vol_sum = df_vol_sum[df_vol_sum['weekday'].isin([4, 5])].copy()
df_vol_sum['月份'] = df_vol_sum['交易日期'].dt.to_period('M')

# 交易量（执行+平仓）
df_vol_sum['交易量'] = df_vol_sum['择时执行交易量'].fillna(0) + df_vol_sum['择时平仓交易量'].fillna(0)

# 这里没有价格，用一个近似汇率：USD约7.3，JPY约0.048
df_vol_sum['交易量_CNY'] = df_vol_sum.apply(
    lambda r: r['交易量'] * 7.3 if r['币种对'] == 'USDCNH' else r['交易量'] * 0.048,
    axis=1
)

vol_monthly_2 = df_vol_sum.groupby('月份')['交易量_CNY'].sum().reset_index()
vol_monthly_2['月份'] = vol_monthly_2['月份'].dt.to_timestamp()

print(f"11-12月交易量数据: {len(vol_monthly_2)} 个月")
vol_monthly_2

In [ ]:
# 合并月度交易量
vol_monthly = pd.concat([vol_monthly_1, vol_monthly_2]).groupby('月份')['交易量_CNY'].sum().reset_index()
vol_monthly = vol_monthly.sort_values('月份').reset_index(drop=True)

# 转换为亿元
vol_monthly['交易量_亿元'] = vol_monthly['交易量_CNY'] / 1e8

print(f"全年交易量数据 {len(vol_monthly)} 个月")
vol_monthly

In [ ]:
# 绘制交易量柱状图
fig7 = go.Figure()

# 阶段分界点：5月（探索期结束）
phase_end = pd.Timestamp('2025-05-01')

# 添加阶段背景色
# 探索期 - 浅黄色
fig7.add_vrect(
    x0=pd.Timestamp('2025-01-01') - pd.Timedelta(days=15),
    x1=phase_end,
    fillcolor='rgba(255, 230, 180, 0.3)',
    layer='below', line_width=0
)
# 扩张期 - 浅绿色
fig7.add_vrect(
    x0=phase_end,
    x1=pd.Timestamp('2025-12-31') + pd.Timedelta(days=15),
    fillcolor='rgba(200, 240, 200, 0.3)',
    layer='below', line_width=0
)

# 分界线
fig7.add_vline(x=phase_end, line_width=1.5, line_dash='dash', line_color='#888888')

# 柱状图
fig7.add_trace(go.Bar(
    x=vol_monthly['月份'],
    y=vol_monthly['交易量_亿元'],
    name='交易量',
    marker_color='#4472C4',
    text=vol_monthly['交易量_亿元'].round(1),
    textposition='outside',
    textfont=dict(size=10)
))

# 折线图
fig7.add_trace(go.Scatter(
    x=vol_monthly['月份'],
    y=vol_monthly['交易量_亿元'],
    mode='lines+markers',
    name='趋势',
    line=dict(color='#4472C4', width=2),
    marker=dict(size=6, color='white', line=dict(color='#4472C4', width=2)),
    showlegend=False
))

# 阶段标签
y_max = vol_monthly['交易量_亿元'].max()
fig7.add_annotation(
    x=pd.Timestamp('2025-03-01'), y=y_max * 1.15,
    text='<b>探索期</b>',
    showarrow=False,
    font=dict(size=13, color='#B8860B'),
    bgcolor='rgba(255,230,180,0.8)',
    borderpad=4
)
fig7.add_annotation(
    x=pd.Timestamp('2025-09-01'), y=y_max * 1.15,
    text='<b>扩张期</b>',
    showarrow=False,
    font=dict(size=13, color='#228B22'),
    bgcolor='rgba(200,240,200,0.8)',
    borderpad=4
)

# 生成每月1号的tick值
all_months = pd.date_range('2025-01-01', '2025-12-01', freq='MS')

fig7.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=all_months,
        ticktext=[d.strftime('%Y-%m') for d in all_months],
        tickangle=-30,
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=False,
        range=[pd.Timestamp('2025-01-01') - pd.Timedelta(days=15), pd.Timestamp('2025-12-31') + pd.Timedelta(days=15)]
    ),
    yaxis=dict(
        title='<b>亿元</b>',
        title_font=dict(size=12),
        tickfont=dict(size=10),
        gridcolor='#E8E8E8',
        showgrid=True,
        range=[0, y_max * 1.3]
    ),
    height=450,
    width=950,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    bargap=0.3,
    margin=dict(l=60, r=60, t=40, b=60)
)

fig7.show()

In [ ]:
# 汇总表格
print("\n" + "="*70)
print("2025年择时策略综合表现")
print("="*70)
print(f"{'策略':<18} {'周数':<8} {'累计损益(万元)':<15} {'周均(元)':<12} {'胜率':<8}")
print("-"*70)

# 周五预锁价-- USD
fri_weeks = len(df_friday)
usd_total = df_friday['USD损益'].sum()
usd_avg = df_friday['USD损益'].mean()
usd_winrate = (df_friday['USD损益'] > 0).sum() / fri_weeks * 100
print(f"{'周五预锁价-USD':<15} {fri_weeks:<8} {usd_total/10000:<15.1f} {usd_avg:<12,.0f} {usd_winrate:<.1f}%")

# 周五预锁价-- JPY
jpy_total = df_friday['JPY损益'].sum()
jpy_avg = df_friday['JPY损益'].mean()
jpy_winrate = (df_friday['JPY损益'] > 0).sum() / fri_weeks * 100
print(f"{'周五预锁价-JPY':<15} {fri_weeks:<8} {jpy_total/10000:<15.1f} {jpy_avg:<12,.0f} {jpy_winrate:<.1f}%")

# 周五预锁价-- 合计
fri_total = df_friday['合计损益'].sum()
fri_avg = df_friday['合计损益'].mean()
fri_winrate = (df_friday['合计损益'] > 0).sum() / fri_weeks * 100
print(f"{'周五预锁价-合计':<14} {fri_weeks:<8} {fri_total/10000:<15.1f} {fri_avg:<12,.0f} {fri_winrate:<.1f}%")

print("-"*70)

# 周一反平
mon_weeks = len(df_monday_all)
mon_total = df_monday_all['损益'].sum()
mon_avg = df_monday_all['损益'].mean()
mon_winrate = (df_monday_all['损益'] > 0).sum() / mon_weeks * 100
print(f"{'周一反平-USD':<15} {mon_weeks:<8} {mon_total/10000:<15.1f} {mon_avg:<12,.0f} {mon_winrate:<.1f}%")

print("="*70)
print(f"{'总计':<18} {'':<8} {(fri_total+mon_total)/10000:<15.1f}")
print("="*70)

In [ ]:
# 保存对比图
fig3.write_html(str(OUTPUT_DIR / "strategy_comparison.html"))
print(f"已保存: {OUTPUT_DIR / 'strategy_comparison.html'}")